# A股主板日线观察池
在收盘后或次日开盘前运行。代码从 GitHub 获取，缓存和跨日状态保存在 Google Drive。

In [ ]:
REPO_URL = "https://github.com/YOUR_NAME/ashare-daily-scanner.git"
BRANCH = "main"
DRIVE_DATA_DIR = "/content/drive/MyDrive/ashare-daily-scanner-data"
RUN_BACKTEST = False
BACKTEST_START = "2024-01-01"
BACKTEST_END = "2025-12-31"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pathlib
import subprocess
import sys

if 'YOUR_NAME' in REPO_URL:
    raise ValueError('请先把 REPO_URL 改成你的 GitHub 仓库地址')

repo_dir = pathlib.Path('/content/ashare-daily-scanner')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(repo_dir)

In [ ]:
command = [
    sys.executable, '-m', 'ashare_scanner',
    '--config', 'config/default.yaml',
    '--data-dir', DRIVE_DATA_DIR,
    'run',
]
subprocess.run(command, check=True)

In [ ]:
import json
import pathlib
import pandas as pd
from IPython.display import display

latest = json.loads((pathlib.Path(DRIVE_DATA_DIR) / 'latest_run.json').read_text(encoding='utf-8'))
run_dir = pathlib.Path(latest['run_dir'])
report = json.loads((run_dir / 'coverage_report.json').read_text(encoding='utf-8'))

print('Run directory:', run_dir)
print('\nSignals summary:')
print(json.dumps(report['signals'], ensure_ascii=False, indent=2))

print('\nWatchlist active:')
watchlist = pd.read_csv(run_dir / 'watchlist_active.csv', dtype={'code': str})
print(f'Total rows: {len(watchlist)}')
display(watchlist.head(100))

print('\nState transitions:')
transitions = pd.read_csv(run_dir / 'state_transitions.csv', dtype={'code': str})
print(f'Total rows: {len(transitions)}')
display(transitions.head(100))

In [ ]:
if RUN_BACKTEST:
    subprocess.run([
        sys.executable, '-m', 'ashare_scanner',
        '--config', 'config/default.yaml',
        '--data-dir', DRIVE_DATA_DIR,
        'backtest', '--start', BACKTEST_START, '--end', BACKTEST_END,
    ], check=True)